<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
0. Import Libraries

</font>
</h2>

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPool1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
1. Data Loading
</font>
</h2>

In [2]:
train = pd.read_csv('torob_train.csv')
test = pd.read_csv('torob_test.csv')


<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
1. Preprocessing
</font>
</h2>

In [ ]:


stopwords = {
    'و', 'در', 'به', 'از', 'که', 'این', 'آن', 'را', 'با', 'برای', 'تا', 'یا',
    'اما', 'اگر', 'هم', 'هر', 'بر', 'پس', 'تو', 'ما', 'شما', 'من', 'او', 'ای',
    'یک', 'دو', 'سه', 'چهار', 'پنج',
    'های', 'ها', 'هایی',
    'است', 'هست', 'بود', 'بوده', 'باشد', 'باشند',
    'شد', 'شده', 'شود', 'شوند',
    'کرد', 'کرده', 'کند', 'کنند',
    'می', 'نمی',
    'دارد', 'دارند', 'داشت', 'داشتند',
    'خواهد', 'خواهند',
    'همه', 'هیچ', 'بعضی', 'دیگر',
    'بیشتر', 'کمتر', 'خیلی', 'بسیار',
    'روی', 'زیر', 'بین', 'میان', 'کنار', 'نزد',
    'مثل', 'مانند', 'بدون', 'درباره', 'توسط',
    'الان', 'اکنون', 'امروز', 'دیروز', 'فردا'
}

prepositions = {
    'از', 'به', 'با', 'در', 'بر', 'برای', 'تا', 'روی', 'زیر',
    'بین', 'میان', 'کنار', 'نزد', 'درباره', 'بدون', 'مثل', 'مانند', 'توی', 'تو'
}

all_stopwords = stopwords.union(prepositions)

def normalize_persian(text):
    text = str(text)
    replacements = {
        'ك': 'ک',
        'ي': 'ی',
        'ى': 'ی',
        'ة': 'ه',
        'ۀ': 'ه',
        'ؤ': 'و',
        'إ': 'ا',
        'أ': 'ا',
        'ٱ': 'ا',
        'ئ': 'ی',
        'ـ': '',
        '\u200c': ' ',
        '\u200f': ' ',
        '\u200e': ' ',
    }
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    return text

def remove_urls_emails_mentions(text):
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'@\S+', ' ', text)
    text = re.sub(r'#\S+', ' ', text)
    return text

def remove_numbers(text):
    return re.sub(r'[0-9۰-۹]+', ' ', text)

def remove_punctuation_and_english(text):
    return re.sub(r'[^آ-ی\s]', ' ', text)

def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

def tokenize(text):
    return text.split()

def remove_stopwords(tokens):
    return [token for token in tokens if token not in all_stopwords]

def remove_short_tokens(tokens, min_len=2):
    return [token for token in tokens if len(token) >= min_len]

def remove_duplicate_tokens(tokens):
    result = []
    for token in tokens:
        if token not in result:
            result.append(token)
    return result

def preprocess_persian_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = normalize_persian(text)
    text = remove_urls_emails_mentions(text)
    text = remove_numbers(text)
    text = remove_punctuation_and_english(text)
    text = remove_extra_spaces(text)

    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = remove_short_tokens(tokens, min_len=2)
    tokens = remove_duplicate_tokens(tokens)

    return ' '.join(tokens)


In [10]:
sample = "خرید 2 عدد کتاب‌های آموزشی جدید در سال 1404 برای کودکان!"
print(preprocess_persian_text(sample))

خرید عدد کتاب آموزشی جدید سال کودکان


In [5]:
train_copy = train.copy()
train_copy = train_copy.drop(columns=['name2'])
train_copy['clean_name'] = train_copy['name1'].apply(preprocess_persian_text)
train_copy = train_copy.drop(columns=['name1'])

In [6]:
train_copy.head()

,cat_id,clean_name
0,0,کاپشن کوهنوردی پر سنگین نورث فیس
1,0,پالتو خزدار مردانه مارک اصلی
2,0,کاپشن مردانه مدل
3,0,کاپشن سالامون پارچه گورتکس کره اعلا ضد آب بادگیر
4,0,کاپشن نظامی آمریکایی سبز


In [28]:
df = train_copy.copy()
df = df.dropna(subset=['clean_name', 'cat_id'])

X = df['clean_name']
y = df['cat_id']


In [29]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
3. Model Development
</font>
</h2>

In [36]:
#Word TF-IDF + LinearSVC
import joblib

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        max_features=30000,
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(
        class_weight='balanced',
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, y_pred))
print("Macro F1:", f1_score(y_valid, y_pred, average='macro'))
print(classification_report(y_valid, y_pred))

final_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        max_features=30000,
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(
        class_weight='balanced',
        random_state=42
    ))
])

final_model.fit(X, y)

joblib.dump(final_model, 'tfidf_linearsvc_model.pkl')


Accuracy: 0.9473087818696884
Macro F1: 0.9476359043602277
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       157
           1       0.90      0.92      0.91       166
           2       0.93      0.96      0.95       166
           3       0.98      0.98      0.98       164
           4       0.97      0.95      0.96       152
           5       0.94      0.93      0.94       165
           6       0.93      0.95      0.94       160
           7       0.99      0.96      0.97       166
           8       0.99      0.99      0.99       150
           9       0.95      0.91      0.93       158
          10       0.91      0.96      0.93       161

    accuracy                           0.95      1765
   macro avg       0.95      0.95      0.95      1765
weighted avg       0.95      0.95      0.95      1765



['tfidf_linearsvc_model.pkl']

In [ ]:

df = train_copy.copy()

df = df.dropna(subset=['clean_name', 'cat_id'])
df['clean_name'] = df['clean_name'].astype(str)
df['cat_id'] = df['cat_id'].astype(int)

X = df['clean_name']
y = df['cat_id']
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
max_words = 30000
max_len = 20

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_valid_seq = tokenizer.texts_to_sequences(X_valid)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_valid_pad = pad_sequences(X_valid_seq, maxlen=max_len, padding='post', truncating='post')


In [8]:
num_classes = df['cat_id'].nunique()
print("num_classes:", num_classes)


num_classes: 11


In [12]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_shape=(max_len,)),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    GlobalMaxPool1D(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.summary()


d:\conda_envs\quera\lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 20, 128)        │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 18, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,898,251 (14.87 MB)

 Trainable params: 3,898,251 (14.87 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
4. Training
</font>
</h2>

In [17]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_valid_pad, y_valid),
    epochs=10,
    batch_size=32,
    #callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9794 - loss: 0.0772 - val_accuracy: 0.9541 - val_loss: 0.1496
Epoch 2/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9820 - loss: 0.0698 - val_accuracy: 0.9547 - val_loss: 0.1529
Epoch 3/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9870 - loss: 0.0533 - val_accuracy: 0.9501 - val_loss: 0.1679
Epoch 4/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9894 - loss: 0.0426 - val_accuracy: 0.9484 - val_loss: 0.1804
Epoch 5/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.9850 - loss: 0.0508 - val_accuracy: 0.9501 - val_loss: 0.1853
Epoch 6/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.9856 - loss: 0.0460 - val_accuracy: 0.9467 - val_loss: 0.1910
Epoch 7/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.9891 - loss: 0.0427 - val_accuracy: 0.9473 - val_loss: 0.2114
Epoch 8/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.9902 - loss: 0.0415 - val_accu

<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
5. Evaluation
</font>
</h2>



In [19]:
y_pred_prob = model.predict(X_valid_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

print("Accuracy:", accuracy_score(y_valid, y_pred))
print("Macro F1:", f1_score(y_valid, y_pred, average='macro'))
print(classification_report(
    y_valid,
    y_pred,
))

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Accuracy: 0.9507082152974504
Macro F1: 0.9510902538954366
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       157
           1       0.93      0.92      0.93       166
           2       0.91      0.97      0.94       166
           3       0.98      0.96      0.97       164
           4       0.99      0.95      0.97       152
           5       0.95      0.97      0.96       165
           6       0.94      0.95      0.95       160
           7       0.99      0.95      0.97       166
           8       0.99      1.00      1.00       150
           9       0.94      0.91      0.92       158
          10       0.90      0.94      0.92       161

    accuracy                           0.95      1765
   macro avg       0.95      0.95      0.95      1765
weighted avg       0.95      0.95      0.95      1765

